# Stage A — FP4 vs FP8 GEMM crossover at M=128 (the MLA decode shape)

**Goal:** Does native FP4 compute beat FP8 at the MLA decode GEMM shape (M=128, K=256–576, N=8K–524K)?

**Prediction (pre-registered):** NO — decode is HBM/work-starved, not compute-bound. FP4's 3× compute
peak doesn't help when the bottleneck is memory/pipeline-fill. The negative is publishable.

**Kill condition:** If FP4 ≤ FP8 TFLOP/s at ALL shapes → publish the negative in Section 5.3.
If FP4 > FP8 at some shapes → proceed to Stage B (accuracy test).

**Hardware:** B300 SXM6 (sm_103a), CUTLASS 4.6, CUDA 12.9.

**Method:** CUTLASS profiler (`cutlass_profiler`) with block-scaled GEMM (mxf8f6f4 format).
Sweep M=128 × K∈{256,512,576} × N∈{1024,8192,32768,131072,524288} × {FP4 E2M1, FP8 E4M3}.
Report TFLOP/s and achieved BW for each.

## 0. Environment setup (B300 vast.ai)

In [ ]:
# Run on a B300 vast.ai instance with CUDA 12.9+
# Image: nvidia/cuda:12.9.0-devel-ubuntu24.04 (or similar with sm_103a support)

import subprocess, os, sys

# Verify GPU
!nvidia-smi --query-gpu=name,compute_cap,memory.total --format=csv,noheader
!nvcc --version | tail -1

In [ ]:
# Clone and build CUTLASS (need the profiler binary)
# If CUTLASS is already installed from the v12 run, skip this cell

CUTLASS_DIR = os.path.expanduser('~/cutlass')
if not os.path.exists(CUTLASS_DIR):
    !git clone --depth 1 --branch v4.6.0 https://github.com/NVIDIA/cutlass.git {CUTLASS_DIR}

BUILD_DIR = os.path.join(CUTLASS_DIR, 'build')
os.makedirs(BUILD_DIR, exist_ok=True)

# Build the profiler with sm_103a support
# Key: enable the block-scaled GEMM kernels (mxf8f6f4)
!cd {BUILD_DIR} && cmake .. \
    -DCUTLASS_NVCC_ARCHS="103a" \
    -DCUTLASS_ENABLE_TESTS=OFF \
    -DCUTLASS_ENABLE_EXAMPLES=OFF \
    -DCUTLASS_ENABLE_PROFILER=ON \
    -DCMAKE_BUILD_TYPE=Release \
    2>&1 | tail -5

!cd {BUILD_DIR} && make cutlass_profiler -j$(nproc) 2>&1 | tail -5

PROFILER = os.path.join(BUILD_DIR, 'tools', 'profiler', 'cutlass_profiler')
assert os.path.exists(PROFILER), f'Profiler not found at {PROFILER}'
print(f'Profiler ready: {PROFILER}')

## 1. The MLA decode GEMM shapes

MLA decode QK matmul: `score = Q @ K^T` where Q is `[M=h_q, D_qk]` and K is `[N_k, D_qk]`.

- **M = 128** (h_q=128 query heads packed — the MLA advantage, no padding needed for NVFP4 M≥128)
- **K = 576** (D_qk = D_latent=512 + D_rope=64 for weight-absorbed) or **K = 512** (latent-only)
- **N = N_k** (context length, sweep 1K→524K)

MLA decode PV matmul: `O = P @ V` where P is `[M=128, N_k]` and V is `[N_k, D_v=512]`.

- **M = 128**, **K = N_k**, **N = 512** (but PV cannot be FP4 — softmax collapses P; proven in v10)

In [ ]:
# Define the sweep shapes
# QK matmul shapes (the ones where FP4 compute could help)
M_VALUES = [128]  # MLA packs all h_q=128 heads
K_VALUES = [256, 512, 576]  # head dim variants
N_VALUES = [1024, 8192, 32768, 131072, 524288]  # context lengths

# Precisions to compare
PRECISIONS = {
    'fp8_e4m3': {'A': 'e4m3', 'B': 'e4m3', 'C': 'f32', 'D': 'f32'},  # FP8 baseline
    'fp4_e2m1': {'A': 'e2m1', 'B': 'e2m1', 'C': 'f32', 'D': 'f32'},  # NVFP4 native
}

print(f'Sweep: {len(M_VALUES)} M × {len(K_VALUES)} K × {len(N_VALUES)} N × {len(PRECISIONS)} precisions')
print(f'= {len(M_VALUES) * len(K_VALUES) * len(N_VALUES) * len(PRECISIONS)} profiler runs')

## 2. Run the CUTLASS profiler sweep

The profiler discovers and benchmarks all available GEMM kernels for each shape/precision.
We filter for block-scaled (`mxf8f6f4`) kernels specifically.

**Alternative if the profiler doesn't expose block-scaled NVFP4 GEMMs directly:**
Use `examples/72_blackwell_narrow_precision_gemm/72b_blackwell_nvfp4_nvfp4_gemm.cu` (the canonical
CUTLASS NVFP4 block-scaled GEMM example for sm_103) — adapt it for M=128 shapes. The Colfax
Research tutorial also covers this path.

In [ ]:
import json, re, time
from collections import defaultdict

results = []

for prec_name, prec_types in PRECISIONS.items():
    for M in M_VALUES:
        for K in K_VALUES:
            for N in N_VALUES:
                print(f'Running: {prec_name} M={M} K={K} N={N}...', end=' ', flush=True)
                
                # CUTLASS profiler command
                # Adjust operation and format flags based on actual CUTLASS 4.6 profiler syntax
                cmd = [
                    PROFILER,
                    '--operation=gemm',
                    f'--m={M}', f'--n={N}', f'--k={K}',
                    f'--A={prec_types["A"]}',
                    f'--B={prec_types["B"]}',
                    f'--C={prec_types["C"]}',
                    f'--D={prec_types["D"]}',
                    '--warmup-iterations=5',
                    '--profiling-iterations=20',
                    '--providers=cutlass',
                    '--output=csv',
                ]
                
                try:
                    result = subprocess.run(cmd, capture_output=True, text=True, timeout=120)
                    if result.returncode == 0:
                        # Parse the CSV output for TFLOP/s and runtime
                        lines = result.stdout.strip().split('\n')
                        # Find the best kernel
                        best_tflops = 0
                        best_line = ''
                        for line in lines[1:]:  # skip header
                            # CSV format varies; look for TFLOP/s column
                            parts = line.split(',')
                            for p in parts:
                                try:
                                    val = float(p)
                                    if val > best_tflops and val < 20000:  # sanity
                                        best_tflops = val
                                        best_line = line
                                except ValueError:
                                    pass
                        
                        results.append({
                            'precision': prec_name,
                            'M': M, 'K': K, 'N': N,
                            'tflops': best_tflops,
                            'raw': best_line[:200],
                        })
                        print(f'{best_tflops:.1f} TFLOP/s')
                    else:
                        print(f'FAILED: {result.stderr[:200]}')
                        results.append({
                            'precision': prec_name,
                            'M': M, 'K': K, 'N': N,
                            'tflops': None,
                            'error': result.stderr[:200],
                        })
                except subprocess.TimeoutExpired:
                    print('TIMEOUT')
                    results.append({
                        'precision': prec_name,
                        'M': M, 'K': K, 'N': N,
                        'tflops': None,
                        'error': 'timeout',
                    })

print(f'\nDone: {len(results)} runs')

## 2b. Fallback: minimal CUTLASS block-scaled GEMM harness

If the profiler doesn't expose E2M1 block-scaled GEMMs, use this minimal C++ harness instead.
This calls the CUTLASS 4.x collective GEMM API directly with block-scaled NVFP4 inputs.

In [ ]:
# Write a minimal CUTLASS block-scaled GEMM benchmark
# This is the fallback if the profiler CLI doesn't handle E2M1 directly

HARNESS_SRC = r'''
#include <cute/tensor.hpp>
#include <cutlass/cutlass.h>
#include <cutlass/gemm/device/gemm_universal_adapter.h>
#include <cutlass/gemm/collective/collective_builder.hpp>
#include <cutlass/epilogue/collective/collective_builder.hpp>
#include <cutlass/numeric_types.h>
#include <cutlass/util/host_tensor.h>
#include <cuda_runtime.h>
#include <stdio.h>
#include <chrono>

// Block-scaled NVFP4 GEMM on sm_103a
// A: E2M1 (4-bit), B: E2M1 (4-bit), C/D: F32
// Block scale factor: E4M3 per 16 elements

using namespace cute;

template <typename ElementA, typename ElementB>
float benchmark_gemm(int M, int N, int K, int warmup, int iters) {
    // Allocate and initialize (random data, FP4 range)
    size_t size_A = M * K;  // in elements (packed for FP4)
    size_t size_B = K * N;
    size_t size_C = M * N;
    
    // ... (allocation + kernel launch + timing)
    // This is a skeleton — the actual collective setup depends on the
    // CUTLASS 4.x API version available on the B300 box.
    
    float total_ms = 0;
    cudaEvent_t start, stop;
    cudaEventCreate(&start);
    cudaEventCreate(&stop);
    
    // Warmup
    for (int i = 0; i < warmup; i++) {
        // run kernel
    }
    cudaDeviceSynchronize();
    
    // Timed
    cudaEventRecord(start);
    for (int i = 0; i < iters; i++) {
        // run kernel
    }
    cudaEventRecord(stop);
    cudaEventSynchronize(stop);
    cudaEventElapsedTime(&total_ms, start, stop);
    
    float avg_ms = total_ms / iters;
    double flops = 2.0 * M * N * K;
    double tflops = flops / (avg_ms * 1e-3) / 1e12;
    
    printf("M=%d N=%d K=%d: %.2f ms, %.1f TFLOP/s\n", M, N, K, avg_ms, tflops);
    
    cudaEventDestroy(start);
    cudaEventDestroy(stop);
    return tflops;
}

int main() {
    printf("Stage A: FP4 vs FP8 GEMM at MLA decode shape (M=128)\n");
    // Shape sweep will be driven from Python; this is the per-shape entry point
    return 0;
}
'''

print('Harness skeleton written. On the B300 box, check which CUTLASS collective API')
print('exposes block-scaled E2M1 GEMMs. The two paths:')
print('  1. cutlass_profiler --A=e2m1 ... (if the profiler handles it)')
print('  2. examples/68_scaled_gemm adapted for M=128 shapes (likely cleaner)')
print('  3. This harness with the specific collective_builder template')

## 3. Analyze results

In [ ]:
import pandas as pd

df = pd.DataFrame(results)
df_valid = df[df['tflops'].notna()].copy()

if len(df_valid) == 0:
    print('No valid results — check profiler output above')
else:
    # Pivot: FP4 vs FP8 at each shape
    pivot = df_valid.pivot_table(
        index=['M', 'K', 'N'],
        columns='precision',
        values='tflops'
    ).reset_index()
    
    if 'fp4_e2m1' in pivot.columns and 'fp8_e4m3' in pivot.columns:
        pivot['fp4_over_fp8'] = pivot['fp4_e2m1'] / pivot['fp8_e4m3']
        pivot['winner'] = pivot.apply(
            lambda r: 'FP4' if r['fp4_over_fp8'] > 1.05 else ('FP8' if r['fp4_over_fp8'] < 0.95 else 'TIE'),
            axis=1
        )
    
    print('\n=== FP4 vs FP8 GEMM crossover at M=128 ===')
    print(pivot.to_string(index=False))
    
    # Verdict
    print('\n=== VERDICT ===')
    if 'winner' in pivot.columns:
        fp4_wins = (pivot['winner'] == 'FP4').sum()
        fp8_wins = (pivot['winner'] == 'FP8').sum()
        ties = (pivot['winner'] == 'TIE').sum()
        print(f'FP4 wins: {fp4_wins}, FP8 wins: {fp8_wins}, Ties: {ties}')
        if fp4_wins == 0:
            print('\n>>> KILL: FP4 never beats FP8 at M=128. Publish the negative.')
            print('>>> The MLA M=128 advantage does NOT convert to a FP4 compute win.')
            print('>>> Decode is HBM/work-starved, not compute-bound — as predicted.')
        elif fp4_wins > 0 and fp8_wins == 0:
            print('\n>>> PROCEED: FP4 beats FP8 at all shapes! Move to Stage B (accuracy).')
        else:
            print(f'\n>>> MIXED: FP4 wins at {fp4_wins} shapes. Analyze the crossover.')

## 4. Roofline context

Map the GEMM results onto the decode roofline to understand WHERE FP4 helps (if at all).

In [ ]:
# Compute the roofline context for each shape
HBM_BW = 8e12  # B300: 8 TB/s
FP8_PEAK = 5e15  # 5 PF
FP4_PEAK = 15e15  # 15 PF

for _, row in df_valid.iterrows():
    M, K, N = row['M'], row['K'], row['N']
    prec = row['precision']
    tflops = row['tflops']
    
    flops = 2 * M * N * K
    # Bytes: A is M×K, B is K×N (element size depends on precision)
    elem_bytes = 0.5 if 'fp4' in prec else 1.0  # FP4=0.5B, FP8=1B
    hbm_bytes = (M * K + K * N + M * N * 4) * elem_bytes  # +output in F32
    ai = flops / hbm_bytes
    
    peak = FP4_PEAK if 'fp4' in prec else FP8_PEAK
    ridge = peak / HBM_BW
    limiter = 'COMPUTE' if ai > ridge else 'HBM'
    pct_peak = (tflops * 1e12 / peak) * 100 if tflops else 0
    
    print(f'{prec} M={M} K={K} N={N}: AI={ai:.1f}, ridge={ridge:.0f}, '
          f'limiter={limiter}, achieved={tflops:.1f} TFLOP/s ({pct_peak:.1f}% peak)')

## 5. Record the result

Whatever the outcome, this data goes into the paper (Section 5.3).

- **Negative (expected):** FP4 ≤ FP8 at M=128. "Native FP4 compute does not help MLA decode."
  The 3× compute peak is wasted because decode is HBM/work-starved. Stage B/C not needed.
  
- **Positive (surprise):** FP4 > FP8 at M=128. Proceed to Stage B (accuracy on real KV).

In [ ]:
# Save results for the paper
if len(df_valid) > 0:
    df_valid.to_csv('stage_a_gemm_results.csv', index=False)
    print('Results saved to stage_a_gemm_results.csv')
    print('\nCopy this data into results.md Stage A and decisions.md Stage A.')